# Dental Vision V1 — post-validation diagnosis
No retraining. Rebuilds the exact DENTEX split, downloads the saved V2 checkpoint from Kaggle Input if attached, evaluates TRAIN vs HOLDOUT at multiple score thresholds, and creates GT/prediction overlays. This tells us whether the failure is overfitting, evaluator/class mapping, or localization.


In [ ]:
!nvidia-smi
import os,shutil,pathlib,json,zipfile,glob
%cd /kaggle/working
shutil.rmtree('/kaggle/working/dental-vision-v1',ignore_errors=True)
!git clone https://github.com/drhaidarali95/dental-vision-v1.git /kaggle/working/dental-vision-v1
%cd /kaggle/working/dental-vision-v1
!pip -q install -r requirements.txt


In [ ]:
# Find the completed 50-epoch V2 checkpoint from attached Kaggle Input.
cands=glob.glob('/kaggle/input/**/dentex_holdout_v2.pt',recursive=True)
print('CHECKPOINT CANDIDATES:',cands)
assert cands, 'Attach the successful Version 12 output as Kaggle Input; dentex_holdout_v2.pt was not found.'
ckpt=cands[0]
print('USING:',ckpt)


In [ ]:
# Rebuild exactly the same 705-image normalized disease dataset and deterministic split.
root=pathlib.Path('data/dentex_diagnose'); shutil.rmtree(root,ignore_errors=True); root.mkdir(parents=True)
!python scripts/download_dentex.py --out data/dentex_diagnose --files training_data.zip
zpath=root/'training_data.zip'; wanted={'caries','deep caries','periapical lesion','periapical lesions','impacted','impacted tooth','impacted teeth'}; candidates=[]
with zipfile.ZipFile(zpath) as z:
    for name in z.namelist():
        if not name.lower().endswith('.json'): continue
        try: d0=json.loads(z.read(name))
        except Exception: continue
        if not isinstance(d0,dict) or not {'images','annotations'}.issubset(d0): continue
        cats=d0.get('categories_3') or d0.get('categories') or []; names={str(c.get('name','')).strip().lower() for c in cats}; score=len(names&wanted); bonus=2 if 'quadrant-enumeration-disease' in name.lower() else 0
        if score or bonus: candidates.append((score+bonus,len(d0['images']),name,d0,cats))
assert candidates
_,_,ann_member,d,cats=max(candidates,key=lambda x:(x[0],x[1])); d['categories']=cats
for a in d['annotations']:
    if 'category_id_3' in a: a['category_id']=a['category_id_3']
subset=root/'diagnostic'; (subset/'images').mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(zpath) as z:
    members=z.namelist()
    for im in d['images']:
        fn=str(im['file_name']).replace('\\','/').lstrip('./'); matches=[m for m in members if m.endswith('/'+fn) or m==fn] or [m for m in members if pathlib.PurePosixPath(m).name==pathlib.PurePosixPath(fn).name]; src=matches[0]; dest=subset/'images'/pathlib.PurePosixPath(fn).name
        with z.open(src) as r,open(dest,'wb') as w: shutil.copyfileobj(r,w)
        im['file_name']=dest.name
(subset/'all.json').write_text(json.dumps(d)); zpath.unlink()
!python scripts/split_dentex_holdout.py --annotations data/dentex_diagnose/diagnostic/all.json --train-out data/dentex_diagnose/diagnostic/train.json --val-out data/dentex_diagnose/diagnostic/val.json --val-fraction 0.20 --seed 20260920


In [ ]:
# Evaluate the SAME checkpoint on training and held-out data. No training.
for split in ['train','val']:
    for thr in [0.001,0.01,0.05]:
        out=f'/kaggle/working/{split}_thr_{thr}.json'
        cmd=f'python evaluate.py --images data/dentex_diagnose/diagnostic/images --annotations data/dentex_diagnose/diagnostic/{split}.json --checkpoint {ckpt} --score-threshold {thr} --output {out}'
        print('\n###',split,'threshold',thr); rc=os.system(cmd); print('exit=',rc)


In [ ]:
# Compact diagnosis table.
rows=[]
for split in ['train','val']:
    for thr in [0.001,0.01,0.05]:
        p=pathlib.Path(f'/kaggle/working/{split}_thr_{thr}.json')
        if p.exists():
            m=json.loads(p.read_text()); rows.append({'split':split,'threshold':thr,'predictions':m['predictions'],'mAP50':m['mAP_50'],'mAP50_95':m['mAP_50_95'],'AR100':m['AR_100']})
print(json.dumps(rows,indent=2))
pathlib.Path('/kaggle/working/diagnosis_summary.json').write_text(json.dumps(rows,indent=2))
print('\nINTERPRETATION: high train/low val = overfit/split generalization; low train+low val = label/geometry/evaluator/model-target problem. Send diagnosis_summary.json to ChatGPT.')
